# Coronagraphic Multi-Resolution HDBSCAN Merge

This notebook demonstrates the current coronagraphic clustering workflow for one SMBHB simulation.

The goal is to identify WDM pixels that are coherent because their coronagraphic `kappa` response has a minimum at a consistent sky patch. Multi-resolution WDM improves the result because the same event can be checked on several time-frequency grids.

## Method Summary

The pipeline has two HDBSCAN stages.

1. Per WDM resolution, select the top 10% A/E WDM pixels in the target frequency band.
2. For every selected pixel, scan HEALPix sky directions and compute the WDM power of coronagraphic `kappa`.
3. Extract the kappa-minimum features per pixel: best sky direction, minimum kappa power, sky contrast, and sky sharpness.
4. Run HDBSCAN within each WDM resolution to cluster pixels with similar kappa-minimum behavior.
5. Summarize each coherent per-resolution cluster into a representative feature vector.
6. Run a second HDBSCAN pass on those representatives to merge clusters that persist across WDM resolutions.

The second stage is the new pipeline feature implemented in `scripts/coronagraphic_multires_merge.py`.

## Environment

Run this notebook from the repository root using the `lisa-tf` environment.

```bash
conda activate lisa-tf
```

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "examples":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RUN_DIR = PROJECT_ROOT / "choronagraphic-cluster" / "runs" / "smbhb_00000_multires_test"
WORK_DIR = PROJECT_ROOT / "cluster" / "runs" / "simulations" / "smbhb_00000"
RUN_DIR

## Optional: Run The Multi-Resolution Coronagraphic Scan

Set `RUN_CORONAGRAPHIC_SCAN = True` only if the multi-resolution run directory does not already exist. This is the expensive step because it computes `kappa` over the sky grid for each WDM layer.

In [ ]:
RUN_CORONAGRAPHIC_SCAN = False

if RUN_CORONAGRAPHIC_SCAN:
    cmd = [
        sys.executable,
        "choronagraphic-cluster/run_one_smbhb.py",
        "--work-dir", str(WORK_DIR.relative_to(PROJECT_ROOT)),
        "--output-dir", str(RUN_DIR.relative_to(PROJECT_ROOT)),
        "--source", "gw_only",
        "--nside", "2",
        "--chunk-duration-s", "8192",
        "--wdm-ms", "128,256,512",
        "--top-pixel-fraction", "0.10",
    ]
    subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)

## Inspect Per-Resolution Results

Each WDM resolution is clustered independently first. The aggregate CSV keeps the physical time and frequency columns, which is necessary because WDM bin indices are not comparable between resolutions.

In [ ]:
summary_path = RUN_DIR / "summary.json"
summary = json.loads(summary_path.read_text())
print(json.dumps({
    "mode": summary.get("mode"),
    "wdm_ms": summary.get("wdm_ms"),
    "n_sky_directions": summary.get("n_sky_directions"),
}, indent=2))

pixels = pd.read_csv(RUN_DIR / "pixel_clusters_all_resolutions.csv")
pixels["wdm_m"] = pixels["wdm_m"].astype(int)
pixels["cluster"] = pixels["cluster"].astype(int)

per_resolution_counts = (
    pixels.groupby("wdm_m")["cluster"]
    .value_counts()
    .sort_index()
    .rename("n_pixels")
    .reset_index()
)
per_resolution_counts.head(30)

## Overlay Multi-Resolution Pixels On A WDM Map

This plot uses a single WDM map as the background and overlays all selected/coherent pixels using physical `time_s` and `freq_hz` coordinates.

In [ ]:
RUN_WDM_OVERLAY = False

if RUN_WDM_OVERLAY:
    cmd = [
        sys.executable,
        "choronagraphic-cluster/plot_multires_wdm_overlay.py",
        "--run-dir", str(RUN_DIR.relative_to(PROJECT_ROOT)),
        "--background-m", "512",
    ]
    subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)

for name in [
    "multires_coherent_overlay_on_wdm.png",
    "multires_selected_overlay_on_wdm.png",
]:
    path = RUN_DIR / "plots" / name
    print(path)
    if path.exists():
        display(Image(filename=str(path)))

## Cross-Resolution HDBSCAN Merge

The merge step groups per-resolution cluster representatives, not raw pixels. Each representative contains:

- median time and frequency,
- median kappa-minimum sky direction,
- median `kappa_min_power`,
- median `sky_contrast`,
- median `sky_sharpness`,
- number of pixels in the per-resolution cluster.

A merged event is stronger if it is supported by at least two WDM resolutions.

In [ ]:
from scripts.coronagraphic_multires_merge import CoronagraphicMergeConfig, run_merge

config = CoronagraphicMergeConfig(
    run_dir=str(RUN_DIR),
    min_cluster_size=2,
    min_resolutions=2,
    make_plots=True,
)
merge_summary = run_merge(config)
merge_summary

## Event-Level Outputs

The most important outputs are:

- `layer_cluster_representatives.csv`: one row per coherent per-resolution cluster.
- `merged_pixel_clusters.csv`: original pixel table plus `event_cluster` labels.
- `multires_event_summary.csv`: event candidates after cross-resolution merging.

`passes_min_resolutions=True` means the candidate is present in at least the requested number of WDM resolutions.

In [ ]:
event_summary = pd.read_csv(RUN_DIR / "multires_event_summary.csv")
event_summary

In [ ]:
representatives = pd.read_csv(RUN_DIR / "layer_cluster_representatives.csv")
representatives[[
    "layer_cluster_id",
    "wdm_m",
    "cluster",
    "n_pixels",
    "median_time_s",
    "median_freq_hz",
    "event_cluster",
    "event_probability",
]].sort_values(["event_cluster", "wdm_m", "cluster"])

## Merge Diagnostics

These plots show the cross-resolution event clusters in physical time-frequency space, in kappa-minimum sky space, and by the number of WDM resolutions supporting each event.

In [ ]:
for name in [
    "multires_event_clusters_time_frequency.png",
    "multires_event_clusters_sky.png",
    "multires_event_resolution_support.png",
]:
    path = RUN_DIR / "plots" / name
    print(path)
    if path.exists():
        display(Image(filename=str(path)))

## Interpretation

For this prototype, a cross-resolution event candidate means:

```text
WDM pixels across several resolutions
-> per-pixel kappa minima point to a consistent sky patch
-> per-resolution HDBSCAN finds coherent groups
-> second-stage HDBSCAN merges those groups across resolutions
```

This is not yet final localization. The current coronagraphic prototype still uses static equilateral LISA geometry, while the simulations use more realistic orbital geometry. The current success criterion is therefore coherent sky-patch behavior and multi-resolution persistence, not exact agreement with the injected RA/Dec.

## Next Practical Checks

Recommended next tests:

1. Increase `nside` from `2` to `4` for a better sky scan.
2. Use `chunk-duration-s 32768` for the fuller event morphology.
3. Compare `gw_only` and `noisy` runs.
4. Track the number of `passes_min_resolutions=True` events as a detection statistic.
5. Replace the static geometry approximation with time-dependent LISA geometry before using sky localization as a performance metric.